# Nebraska K–12 Chronic Absenteeism — District-Level ML Analysis
## Final Report Notebook

---

| | |
|:---|:---|
| **Author** | Abhinav Adhikari |
| **Course** | DSCI/STAT 8950 — Data Science Capstone |
| **Institution** | University of Nebraska Omaha |
| **Semester** | Spring 2026 |
| **Data Source** | Nebraska Department of Education (NDE) — Public Data Archive |
| **Scope** | District-level analysis — 244 Nebraska school districts, 7 years (2018–19 through 2024–25) |

---

### Project Summary

This notebook predicts district-level chronic absenteeism rates across Nebraska public schools using 7 years of NDE data.
Six gradient-boosting models are trained, tuned, and compared. An additional **safe-feature early-intervention model**
is built using only features available before the school year begins — enabling prospective district risk scoring.

| Headline Metric | Value |
|:----------------|:------|
| Best model | XGBoost (Tuned) |
| Test RMSE | **~1.68 percentage points** |
| Test R² | **~0.951** |
| Districts | 244 |
| District-years | 1,715 |
| Years | 7 (2018–19 through 2024–25) |
| Models compared | 6 (XGBoost, Random Forest, LightGBM — default + tuned each) |

## Table of Contents

| # | Section |
|:--|:--------|
| 1 | Imports |
| 2 | Data Loading — 3 sources, 7 years, 1 715 district-year observations |
| 3 | Feature Engineering — 9 features |
| 4 | Exploratory Data Analysis — COVID trend, FRL correlation |
| 5 | Train / Test Split — 80/20 stratified by year |
| 6 | XGBoost — Default & Tuned |
| 7 | Random Forest — Default & Tuned |
| 8 | LightGBM — Default & Tuned |
| 9 | Model Comparison — All 6 models side-by-side |
| 10 | Overfitting Check — Train vs Test R² gap |
| 11 | Feature Importance — XGBoost Tuned (best model) |
| 12 | Safe-Feature Early-Intervention Model — 7 advance-available features |
| 13 | Per-Year Generalization Check |
| 14 | Final Summary & Report Numbers |

## 1. Imports

In [ ]:
!pip install xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn scipy openpyxl xlrd -q
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# ── Consistent dark-navy palette ──────────────────────────────────────────
# ── Espresso palette — matches final presentation ────────────────────────
NAVY   = '#0C0806'; BLUE   = '#2d7dd2'; TEAL   = '#1A936F'
AMBER  = '#C8820A'; RED    = '#E05050'; GREEN  = '#38B272'
MUTED  = '#A89070'; PURPLE = '#6C4FCC'; CREAM  = '#FBF2DC'

plt.rcParams.update({
    'figure.facecolor': NAVY,   'axes.facecolor':   '#221608',
    'axes.edgecolor':   '#382818', 'axes.labelcolor': MUTED,
    'xtick.color':      MUTED,  'ytick.color':      MUTED,
    'text.color':       CREAM, 'grid.color':        '#382818',
    'grid.linewidth':   0.5,    'axes.spines.top':  False,
    'axes.spines.right':False,  'font.size':        11,
    'axes.titlesize':   13,     'axes.titleweight': 'bold',
    'figure.dpi':       110,
})
print("Imports OK")

## 2. Data Loading — 3 Sources · 7 Years · District-Level Panel

Three NDE data series are merged per year:

| # | Source | Files | Purpose |
|:--|:-------|:------|:--------|
| 1 | Chronic Absence Reports | `absence_YYYY-YY.xlsx` | Target variable + in-year signals |
| 2 | Free & Reduced Lunch (FRL) | `frl_YYYY-YY.xlsx/.xls` | Primary socioeconomic proxy |
| 3 | Grade-level Membership | Same absence file, `BY GRADE` sheet | High-school enrollment fraction |

All data is publicly available from the Nebraska Department of Education — no login required.
Districts with fewer than 50 enrolled students are excluded (unreliable rates at micro-district scale).

In [ ]:
# ── Locate data folder ────────────────────────────────────────────────────
_probe = 'absence_2018-19.xlsx'
DATA_DIR = 'data'
for _d in ['data', '.', '/mnt/user-data/uploads']:
    if os.path.exists(os.path.join(_d, _probe)):
        DATA_DIR = _d
        break
print(f"DATA_DIR = {DATA_DIR}")

ALL_YEARS = ['2018-19','2019-20','2020-21','2021-22','2022-23','2023-24','2024-25']

ABSENCE_FILES = {yr: os.path.join(DATA_DIR, f'absence_{yr}.xlsx') for yr in ALL_YEARS}
FRL_FILES = {
    '2018-19': os.path.join(DATA_DIR, 'frl_2018-19.xls'),
    '2019-20': os.path.join(DATA_DIR, 'frl_2019-20.xls'),
    '2020-21': os.path.join(DATA_DIR, 'frl_2020-21.xls'),
    '2021-22': os.path.join(DATA_DIR, 'frl_2021-22.xlsx'),
    '2022-23': os.path.join(DATA_DIR, 'frl_2022-23.xlsx'),
    '2023-24': os.path.join(DATA_DIR, 'frl_2023-24.xlsx'),
    '2024-25': os.path.join(DATA_DIR, 'frl_2024-25.xlsx'),
}

In [ ]:
def load_district(path, year):
    df = pd.read_excel(path, sheet_name='DISTRICT TOTAL', header=4)
    df.columns = ['DISTRICT_CODE','DISTRICT_NAME','MEMBERSHIP',
                  'ABS10','ABS15','ABS20','CHRONIC_N','PCT20']
    df = df[df['MEMBERSHIP'].apply(
        lambda x: str(x).replace('.','').replace(' ','').isdigit())]
    for c in ['MEMBERSHIP','ABS10','ABS15','ABS20','CHRONIC_N']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=['MEMBERSHIP','CHRONIC_N'])
    df = df[df['MEMBERSHIP'] >= 50]
    df['CHRONIC_RATE'] = df['CHRONIC_N'] / df['MEMBERSHIP']
    df['ABS10_RATE']   = df['ABS10']     / df['MEMBERSHIP']
    df['ABS15_RATE']   = df['ABS15']     / df['MEMBERSHIP']
    df['YEAR'] = year
    df['DIST_KEY'] = (df['DISTRICT_CODE'].astype(str)
                      .str.replace(r'[^0-9]','',regex=True)
                      .str.lstrip('0').str[:6])
    return df

def load_frl(path, year):
    engine = 'xlrd' if path.endswith('.xls') else 'openpyxl'
    hdr    = 4 if path.endswith('.xls') else 0
    df = pd.read_excel(path, sheet_name='Districts Data ', header=hdr, engine=engine)
    df.columns = [c.strip() for c in df.columns]
    df['FRL_PCT']  = pd.to_numeric(df['PERCENT'], errors='coerce')
    df['DIST_KEY'] = (df['CODISTSCH'].astype(str)
                      .str.replace(r'[^0-9]','',regex=True)
                      .str.lstrip('0').str[:6])
    df['_dn'] = df['CODISTSCH'].astype(str).str[-3:].str.replace(r'[^0-9]','',regex=True)
    df = df[pd.to_numeric(df['_dn'], errors='coerce') < 700]
    return df.groupby('DIST_KEY')['FRL_PCT'].mean().reset_index()

def load_hs_pct(path):
    df = pd.read_excel(path, sheet_name='BY GRADE', header=4)
    df.columns = ['DISTRICT_CODE','DISTRICT_NAME','GRADE','MEMBERSHIP',
                  'ABS10','ABS15','ABS20','CHRONIC_N','PCT20']
    df['MEMBERSHIP'] = pd.to_numeric(df['MEMBERSHIP'], errors='coerce')
    df['DIST_KEY'] = (df['DISTRICT_CODE'].astype(str)
                      .str.replace(r'[^0-9]','',regex=True)
                      .str.lstrip('0').str[:6])
    df['GRADE'] = df['GRADE'].astype(str).str.strip()
    def hs_frac(g):
        tot = g['MEMBERSHIP'].sum()
        hs  = g[g['GRADE'].isin(['09','10','11','12'])]['MEMBERSHIP'].sum()
        return hs / tot if tot > 0 else np.nan
    return (df.groupby('DIST_KEY').apply(hs_frac)
              .rename('HS_ENROLL_PCT').reset_index())

In [ ]:
# ── Stack all 7 years ─────────────────────────────────────────────────────
frames = []
print(f"{'Year':<10} {'Districts':>10} {'FRL':>8} {'Merged':>8}")
print("-" * 42)
for year in ALL_YEARS:
    ab = load_district(ABSENCE_FILES[year], year)
    fr = load_frl(FRL_FILES[year], year)
    hs = load_hs_pct(ABSENCE_FILES[year])
    mg = ab.merge(fr, on='DIST_KEY', how='inner').merge(hs, on='DIST_KEY', how='left')
    frames.append(mg)
    print(f"{year:<10} {len(ab):>10} {len(fr):>8} {len(mg):>8}")

panel = pd.concat(frames, ignore_index=True)
panel = panel.dropna(subset=['CHRONIC_RATE','FRL_PCT'])
panel = panel[panel['MEMBERSHIP'] >= 50]
print(f"\nPanel : {len(panel)} rows  |  {panel['DIST_KEY'].nunique()} unique districts  |  {panel['YEAR'].nunique()} years")

## 3. Feature Engineering

| Feature | Available | Rationale |
|:--------|:----------|:----------|
| `FRL_PCT` | **Pre-year** | Poverty proxy — strongest advance-available predictor |
| `LOG_MEMBERSHIP` | **Pre-year** | District size on log scale |
| `ABS10_RATE` | Mid-year | 10+ day absence rate — early warning signal |
| `ABS15_RATE` | Mid-year | 15+ day absence rate — strongest predictor overall |
| `HS_ENROLL_PCT` | **Pre-year** | High-school fraction — Grade 8→9 escalation |
| `IS_URBAN` | **Pre-year** | Douglas (Omaha) or Lancaster (Lincoln) county flag |
| `SIZE_TIER` | **Pre-year** | Enrollment quartile 1–4 |
| `FRL_SQ` | **Pre-year** | FRL%² — nonlinear poverty acceleration |
| `YEAR_NUM` | **Pre-year** | 0–6 for 2018-19→2024-25; encodes COVID trend |

**7 of the 9 features are available before the school year begins** — enabling the safe-feature early-intervention model in Section 12.

In [ ]:
panel['YEAR_NUM']       = panel['YEAR'].map({yr: i for i, yr in enumerate(ALL_YEARS)})
panel['LOG_MEMBERSHIP'] = np.log1p(panel['MEMBERSHIP'])
panel['COUNTY_NUM']     = pd.to_numeric(
    panel['DISTRICT_CODE'].astype(str).str.split('-').str[0].str.lstrip('0'), errors='coerce')
panel['IS_URBAN']       = panel['COUNTY_NUM'].isin([28, 55]).astype(int)
panel['FRL_SQ']         = panel['FRL_PCT'] ** 2
panel['SIZE_TIER']      = pd.qcut(panel['MEMBERSHIP'], q=4, labels=[1,2,3,4]).astype(int)

# Full feature set (9 features — includes mid-year ABS rates)
FEATURES_FULL = ['FRL_PCT','LOG_MEMBERSHIP','ABS10_RATE','ABS15_RATE',
                 'HS_ENROLL_PCT','IS_URBAN','SIZE_TIER','FRL_SQ','YEAR_NUM']

# Safe features (7 advance-available — no ABS rates)
FEATURES_SAFE = ['FRL_PCT','LOG_MEMBERSHIP','HS_ENROLL_PCT',
                 'IS_URBAN','SIZE_TIER','FRL_SQ','YEAR_NUM']

TARGET = 'CHRONIC_RATE'

ml = panel[FEATURES_FULL + [TARGET,'DISTRICT_NAME','MEMBERSHIP','YEAR','DIST_KEY']].dropna()
X  = ml[FEATURES_FULL].values
y  = ml[TARGET].values

print(f"Full dataset  : {len(ml)} rows x {len(FEATURES_FULL)} features")
print(f"Districts     : {ml['DIST_KEY'].nunique()} unique")
print(f"Target range  : {y.min()*100:.1f}% – {y.max()*100:.1f}%  mean={y.mean()*100:.1f}%")
ml[FEATURES_FULL + [TARGET]].describe().round(3)

## 4. Exploratory Data Analysis

Key patterns the models must learn:
- **2021–22 is the COVID peak** (20.1% mean chronic rate — highest in 7 years)
- **FRL% drives absenteeism** at district level across all years (r ≈ 0.73)
- **Recovery is incomplete** — 2024–25 at 13.1%, still 54% above 2018–19 baseline of 8.5%

In [ ]:
YEAR_COLORS = {
    '2018-19':'#7b5ea7','2019-20':MUTED,
    '2020-21':BLUE, '2021-22':RED,
    '2022-23':AMBER,'2023-24':TEAL,'2024-25':GREEN
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Distribution by year
for yr, clr in YEAR_COLORS.items():
    sub = ml[ml['YEAR'] == yr]['CHRONIC_RATE'] * 100
    axes[0].hist(sub, bins=25, alpha=0.5, color=clr,
                 label=f"{yr} ({sub.mean():.1f}%)", edgecolor='none')
axes[0].set_xlabel('Chronic Rate (%)'); axes[0].set_ylabel('Districts')
axes[0].set_title('Chronic Rate Distribution\n2018-19 through 2024-25')
axes[0].legend(fontsize=7)

# FRL vs Chronic (all 7 years)
axes[1].scatter(ml['FRL_PCT']*100, y*100, color=TEAL, alpha=0.2, s=18, edgecolors='none')
m_, b_, r_, *_ = scipy_stats.linregress(ml['FRL_PCT']*100, y*100)
xl = np.linspace(0, 100, 100)
axes[1].plot(xl, m_*xl+b_, color=AMBER, lw=2, label=f'r = {r_:.3f}')
axes[1].set_xlabel('FRL %'); axes[1].set_ylabel('Chronic Rate (%)')
axes[1].set_title(f'FRL% vs Chronic Rate\n(r = {r_:.3f}, all 7 years)')
axes[1].legend(fontsize=9)

# Year-over-year trend
yr_mu = ml.groupby('YEAR')['CHRONIC_RATE'].mean() * 100
yr_sd = ml.groupby('YEAR')['CHRONIC_RATE'].std()  * 100
xs = list(range(len(ALL_YEARS)))
ys_m = [yr_mu[y] for y in ALL_YEARS]
ys_s = [yr_sd[y] for y in ALL_YEARS]
axes[2].plot(xs, ys_m, 'o-', color=BLUE, lw=2.5, markersize=8,
             markerfacecolor='white', markeredgewidth=2.5, markeredgecolor=BLUE)
axes[2].fill_between(xs, [m-s for m,s in zip(ys_m,ys_s)],
                         [m+s for m,s in zip(ys_m,ys_s)], alpha=0.12, color=BLUE)
for i, yr in enumerate(ALL_YEARS):
    axes[2].annotate(f"{yr_mu[yr]:.1f}%", (i, yr_mu[yr]),
                     textcoords='offset points', xytext=(0, 10),
                     ha='center', fontsize=9, color='white')
axes[2].set_xticks(xs); axes[2].set_xticklabels(ALL_YEARS, rotation=30, ha='right')
axes[2].set_ylabel('Mean Chronic Rate (%)')
axes[2].set_title('Year-over-Year Trend')
axes[2].axvline(3, color=RED, lw=1.2, ls='--', alpha=0.5, label='COVID peak (2021-22)')
axes[2].legend(fontsize=8)

plt.suptitle(f'Panel EDA — {len(ml)} District-Year Observations', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

print("Mean chronic rate by year:")
for yr in ALL_YEARS:
    print(f"  {yr}: {yr_mu[yr]:.1f}%")

**EDA Key Findings:**
1. **2021–22 is the COVID peak** — mean chronic rate jumped to its highest level across all 7 years
2. **Post-COVID recovery is incomplete** — 2024–25 remains well above the 2018–19 baseline
3. **FRL% is consistently the primary structural driver** (r ≈ 0.73) — confirmed across all 7 years
4. **Wide within-year variation** (SD ~7 pp) — the FRL/absence features explain cross-district differences beyond what year alone captures

## 5. Train / Test Split

**80/20 stratified by year** — ensures proportional representation of every year (including the COVID peak) in both training and test sets. Test set is **never used during model tuning**.

In [ ]:
np.random.seed(42)
X_tr, X_te, y_tr, y_te, yr_tr, yr_te = train_test_split(
    X, y, ml['YEAR'].values, test_size=0.20, random_state=42,
    stratify=ml['YEAR'].values)

print(f"Train : {len(y_tr)} rows  |  Test : {len(y_te)} rows")
print(f"\n{'Year':<10} {'Train':>8} {'Test':>8}")
print("-" * 30)
for yr in ALL_YEARS:
    print(f"{yr:<10} {(yr_tr==yr).sum():>8} {(yr_te==yr).sum():>8}")
print(f"\nTrain mean: {y_tr.mean()*100:.2f}%  |  Test mean: {y_te.mean()*100:.2f}%")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

## 6. XGBoost — Default & Tuned

XGBoost builds sequential shallow trees, each correcting the residuals of the previous one. Built-in L1/L2 regularization prevents extreme memorization even at defaults.

In [ ]:
# ── XGBoost Default ───────────────────────────────────────────────────────
xgb_def = XGBRegressor(random_state=42, verbosity=0)
xgb_def.fit(X_tr, y_tr)

xgb_def_tr = np.sqrt(mean_squared_error(y_tr, xgb_def.predict(X_tr))) * 100
xgb_def_te = np.sqrt(mean_squared_error(y_te, xgb_def.predict(X_te))) * 100
xgb_def_r2_tr = r2_score(y_tr, xgb_def.predict(X_tr))
xgb_def_r2_te = r2_score(y_te, xgb_def.predict(X_te))
print(f"XGBoost Default  — Train RMSE: {xgb_def_tr:.3f}%  Test RMSE: {xgb_def_te:.3f}%  Train R²: {xgb_def_r2_tr:.4f}  Test R²: {xgb_def_r2_te:.4f}")

# ── XGBoost Tuned (GridSearchCV) ───────────────────────────────────────────
xgb_gs = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    {'n_estimators':[100,200,300], 'max_depth':[2,3,4],
     'learning_rate':[0.03,0.05,0.1], 'subsample':[0.8,1.0],
     'colsample_bytree':[0.8,1.0]},
    cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=0)
xgb_gs.fit(X_tr, y_tr)

xgb_best = xgb_gs.best_estimator_
y_pred_xgb = xgb_best.predict(X_te)
xgb_tr_r2  = r2_score(y_tr, xgb_best.predict(X_tr))
xgb_te_r2  = r2_score(y_te, y_pred_xgb)
xgb_tr_rmse = np.sqrt(mean_squared_error(y_tr, xgb_best.predict(X_tr))) * 100
xgb_te_rmse = np.sqrt(mean_squared_error(y_te, y_pred_xgb)) * 100
xgb_te_mae  = mean_absolute_error(y_te, y_pred_xgb) * 100

print(f"XGBoost Tuned    — Train RMSE: {xgb_tr_rmse:.3f}%  Test RMSE: {xgb_te_rmse:.3f}%  Train R²: {xgb_tr_r2:.4f}  Test R²: {xgb_te_r2:.4f}")
print(f"  Best params: {xgb_gs.best_params_}")
print(f"  Test MAE: {xgb_te_mae:.3f}%")

## 7. Random Forest — Default & Tuned

Random Forest builds parallel independent trees and averages their predictions. Less prone to overfitting than single trees but generally weaker than gradient boosting on tabular data.

In [ ]:
# ── Random Forest Default ─────────────────────────────────────────────────
rf_def = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_def.fit(X_tr, y_tr)

rf_def_tr = np.sqrt(mean_squared_error(y_tr, rf_def.predict(X_tr))) * 100
rf_def_te = np.sqrt(mean_squared_error(y_te, rf_def.predict(X_te))) * 100
rf_def_r2_tr = r2_score(y_tr, rf_def.predict(X_tr))
rf_def_r2_te = r2_score(y_te, rf_def.predict(X_te))
print(f"RF Default       — Train RMSE: {rf_def_tr:.3f}%  Test RMSE: {rf_def_te:.3f}%  Train R²: {rf_def_r2_tr:.4f}  Test R²: {rf_def_r2_te:.4f}")

# ── Random Forest Tuned ────────────────────────────────────────────────────
rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {'n_estimators':[100,200,300], 'max_depth':[None,8,12],
     'min_samples_leaf':[1,3,5], 'max_features':['sqrt',0.7]},
    cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=0)
rf_gs.fit(X_tr, y_tr)

rf_best = rf_gs.best_estimator_
y_pred_rf = rf_best.predict(X_te)
rf_tr_r2   = r2_score(y_tr, rf_best.predict(X_tr))
rf_te_r2   = r2_score(y_te, y_pred_rf)
rf_tr_rmse = np.sqrt(mean_squared_error(y_tr, rf_best.predict(X_tr))) * 100
rf_te_rmse = np.sqrt(mean_squared_error(y_te, y_pred_rf)) * 100

print(f"RF Tuned         — Train RMSE: {rf_tr_rmse:.3f}%  Test RMSE: {rf_te_rmse:.3f}%  Train R²: {rf_tr_r2:.4f}  Test R²: {rf_te_r2:.4f}")
print(f"  Best params: {rf_gs.best_params_}")

## 8. LightGBM — Default & Tuned

LightGBM uses leaf-wise tree growth (vs depth-wise in XGBoost), often faster on larger datasets. Requires careful tuning to prevent overfitting on smaller datasets like this one.

In [ ]:
# ── LightGBM Default ──────────────────────────────────────────────────────
lgb_def = LGBMRegressor(random_state=42, verbosity=-1)
lgb_def.fit(X_tr, y_tr)

lgb_def_tr = np.sqrt(mean_squared_error(y_tr, lgb_def.predict(X_tr))) * 100
lgb_def_te = np.sqrt(mean_squared_error(y_te, lgb_def.predict(X_te))) * 100
lgb_def_r2_tr = r2_score(y_tr, lgb_def.predict(X_tr))
lgb_def_r2_te = r2_score(y_te, lgb_def.predict(X_te))
print(f"LightGBM Default — Train RMSE: {lgb_def_tr:.3f}%  Test RMSE: {lgb_def_te:.3f}%  Train R²: {lgb_def_r2_tr:.4f}  Test R²: {lgb_def_r2_te:.4f}")

# ── LightGBM Tuned ─────────────────────────────────────────────────────────
lgb_gs = GridSearchCV(
    LGBMRegressor(random_state=42, verbosity=-1),
    {'n_estimators':[100,200,300], 'max_depth':[3,4,6],
     'learning_rate':[0.03,0.05,0.1], 'num_leaves':[15,31,63],
     'min_child_samples':[5,10,20]},
    cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=0)
lgb_gs.fit(X_tr, y_tr)

lgb_best = lgb_gs.best_estimator_
y_pred_lgb = lgb_best.predict(X_te)
lgb_tr_r2   = r2_score(y_tr, lgb_best.predict(X_tr))
lgb_te_r2   = r2_score(y_te, y_pred_lgb)
lgb_tr_rmse = np.sqrt(mean_squared_error(y_tr, lgb_best.predict(X_tr))) * 100
lgb_te_rmse = np.sqrt(mean_squared_error(y_te, y_pred_lgb)) * 100

print(f"LightGBM Tuned   — Train RMSE: {lgb_tr_rmse:.3f}%  Test RMSE: {lgb_te_rmse:.3f}%  Train R²: {lgb_tr_r2:.4f}  Test R²: {lgb_te_r2:.4f}")
print(f"  Best params: {lgb_gs.best_params_}")

## 9. Model Comparison — All 6 Gradient-Boosting Models

Side-by-side comparison on Test RMSE and Test R². This is the primary results figure for the final report.

In [ ]:
model_results = {
    'XGBoost\nDefault':  (xgb_def.predict(X_te),   xgb_def.predict(X_tr)),
    'XGBoost\nTuned':    (y_pred_xgb,               xgb_best.predict(X_tr)),
    'Random Forest\nDefault': (rf_def.predict(X_te), rf_def.predict(X_tr)),
    'Random Forest\nTuned':   (y_pred_rf,             rf_best.predict(X_tr)),
    'LightGBM\nDefault': (lgb_def.predict(X_te),    lgb_def.predict(X_tr)),
    'LightGBM\nTuned':   (y_pred_lgb,               lgb_best.predict(X_tr)),
}

names    = list(model_results.keys())
te_rmses = [np.sqrt(mean_squared_error(y_te, v[0]))*100 for v in model_results.values()]
tr_rmses = [np.sqrt(mean_squared_error(y_tr, v[1]))*100 for v in model_results.values()]
te_r2s   = [r2_score(y_te, v[0])                         for v in model_results.values()]
tr_r2s   = [r2_score(y_tr, v[1])                         for v in model_results.values()]

colors = [MUTED, TEAL, MUTED, BLUE, MUTED, AMBER]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test RMSE
axes[0].barh(names, te_rmses, color=colors, alpha=0.85, height=0.5, edgecolor='none')
for i, v in enumerate(te_rmses):
    axes[0].text(v+0.04, i, f'{v:.3f}%', va='center', fontsize=9, color='white')
axes[0].set_xlabel('Test RMSE (pp)'); axes[0].set_title('Test RMSE — lower is better')
axes[0].axvline(min(te_rmses), color=GREEN, lw=1.5, ls='--', alpha=0.7)

# Test R²
axes[1].barh(names, te_r2s, color=colors, alpha=0.85, height=0.5, edgecolor='none')
for i, v in enumerate(te_r2s):
    axes[1].text(v+0.003, i, f'{v:.4f}', va='center', fontsize=9, color='white')
axes[1].set_xlabel('Test R²'); axes[1].set_title('Test R² — higher is better')
axes[1].axvline(max(te_r2s), color=GREEN, lw=1.5, ls='--', alpha=0.7)

plt.suptitle('All 6 Gradient-Boosting Models — Test Set Performance', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

print(f"\n{'Model':<25} {'Train RMSE':>12} {'Test RMSE':>12} {'Train R²':>10} {'Test R²':>10}")
print("-" * 72)
for nm, tr_r, te_r, tr_r2, te_r2 in zip(names, tr_rmses, te_rmses, tr_r2s, te_r2s):
    star = ' ★' if te_r == min(te_rmses) else ''
    print(f"  {nm.replace(chr(10),' '):<23} {tr_r:>11.3f}%  {te_r:>11.3f}%  {tr_r2:>9.4f}  {te_r2:>9.4f}{star}")

## 10. Overfitting Check — Train vs Test R² Gap

A large Train R² with a small Test R² signals overfitting — the model memorized noise rather than learning patterns. This matches the analysis presented in the final presentation (Slide 11).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

short_names = ['XGBoost\nDefault','XGBoost\nTuned','RF\nDefault',
               'RF\nTuned','LightGBM\nDefault','LightGBM\nTuned']
x = np.arange(len(short_names))
w = 0.35

bars1 = ax.bar(x - w/2, tr_r2s, w, label='Train R²', color=BLUE,  alpha=0.85, edgecolor='none')
bars2 = ax.bar(x + w/2, te_r2s, w, label='Test R²',  color=AMBER, alpha=0.85, edgecolor='none')

for b1, b2, tr, te in zip(bars1, bars2, tr_r2s, te_r2s):
    gap = tr - te
    ax.text(b1.get_x()+b1.get_width()/2, b1.get_height()+0.005, f'{tr:.3f}',
            ha='center', va='bottom', fontsize=8, color='white')
    ax.text(b2.get_x()+b2.get_width()/2, b2.get_height()+0.005, f'{te:.3f}',
            ha='center', va='bottom', fontsize=8, color='white')
    if gap > 0.03:
        ax.annotate(f'gap\n{gap:.3f}', xy=(b2.get_x()+b2.get_width()/2, te),
                    xytext=(b2.get_x()+b2.get_width()/2, te-0.08),
                    ha='center', fontsize=7, color=RED,
                    arrowprops=dict(arrowstyle='->', color=RED, lw=0.8))

ax.set_xticks(x); ax.set_xticklabels(short_names, fontsize=9)
ax.set_ylabel('R²'); ax.set_ylim(0.7, 1.05)
ax.set_title('Train vs Test R² — Overfitting Check')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
ax.axhline(1.0, color='white', lw=0.5, ls=':')
plt.tight_layout(); plt.show()

**Overfitting analysis:**
- **XGBoost (Tuned) has the smallest train-test gap** — best generalization across all 6 models
- LightGBM defaults tend to overfit on this dataset (Train R² ~0.99, Test R² lower) — leaf-wise growth needs regularization
- Tuned models consistently close the gap: GridSearchCV penalizes splits that don't improve CV RMSE
- Random Forest defaults show moderate overfitting — bagging helps but doesn't eliminate it

## 11. Feature Importance — XGBoost Tuned (Best Model)

Feature importance from the best-performing model. This matches the district-level feature importance panel on Slide 12 of the final presentation.

In [ ]:
feat_labels = ['FRL%','Log(Enroll)','10+Rate','15+Rate',
               'HS%','Urban','SizeTier','FRL\u00b2','Year']

imp = xgb_best.feature_importances_
order = np.argsort(imp)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].barh([feat_labels[i] for i in order], [imp[i] for i in order],
             color=TEAL, alpha=0.85, height=0.55, edgecolor='none')
for i, idx in enumerate(order):
    axes[0].text(imp[idx]+0.005, i, f'{imp[idx]*100:.1f}%',
                 va='center', fontsize=9, color='white')
axes[0].set_xlabel('Feature Importance (fraction)')
axes[0].set_title('XGBoost Tuned — Feature Importance')

# Pie chart for top features
top_idx = np.argsort(imp)[::-1]
top_labels = [feat_labels[i] for i in top_idx]
top_vals   = [imp[i] for i in top_idx]
colors_pie = [TEAL, BLUE, AMBER, GREEN, PURPLE, RED, MUTED, '#5ba3d9', '#e8c07d']
wedges, texts, autotexts = axes[1].pie(
    top_vals, labels=top_labels, autopct='%1.1f%%',
    colors=colors_pie, startangle=90,
    textprops={'color':'white','fontsize':9})
axes[1].set_title('XGBoost Tuned — Importance Distribution')
axes[1].set_facecolor(NAVY)

plt.suptitle('Feature Importance — District-Level XGBoost (Tuned)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

print("Feature importance ranking:")
for i in np.argsort(imp)[::-1]:
    print(f"  {feat_labels[i]:<15} {imp[i]*100:>6.1f}%")

**Feature importance findings (matching Slide 12 of final presentation):**

| Tier | Feature | Role |
|:-----|:--------|:-----|
| **Dominant** | `ABS15_RATE` (~52%) | Mid-year signal — once 15% of students miss 15 days, chronic outcome is near-certain |
| **Strong** | `ABS10_RATE` (~24%) | Earlier warning — available mid-year before ABS15 accumulates |
| **Pre-year** | `FRL_PCT` (~11%) | Strongest advance-available predictor — use for prospective risk scoring |
| **Pre-year** | `FRL_SQ`, `HS%`, `Year` | Supporting structural features |

**Policy insight:** Two early-warning strategies emerge — (1) before the year: rank districts by FRL%; (2) mid-year: flag districts where ABS10/ABS15 rates are climbing.

## 12. Safe-Feature Early-Intervention Model

This section builds a model using **only the 7 features available before the school year begins** — removing `ABS10_RATE` and `ABS15_RATE` which are mid-year measurements.

This enables NDE to generate a **district risk score at the start of each school year**, before any attendance data is collected. Superintendents can proactively allocate resources to high-risk districts in September rather than waiting until February or March.

**Safe features (7):** `FRL_PCT`, `LOG_MEMBERSHIP`, `HS_ENROLL_PCT`, `IS_URBAN`, `SIZE_TIER`, `FRL_SQ`, `YEAR_NUM`

In [ ]:
# ── Safe-feature dataset ─────────────────────────────────────────────────
ml_safe = panel[FEATURES_SAFE + [TARGET,'DISTRICT_NAME','MEMBERSHIP','YEAR','DIST_KEY']].dropna()
X_safe  = ml_safe[FEATURES_SAFE].values
y_safe  = ml_safe[TARGET].values

X_tr_s, X_te_s, y_tr_s, y_te_s, yr_tr_s, yr_te_s = train_test_split(
    X_safe, y_safe, ml_safe['YEAR'].values,
    test_size=0.20, random_state=42, stratify=ml_safe['YEAR'].values)

# ── Tune XGBoost on safe features ─────────────────────────────────────────
xgb_safe_gs = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    {'n_estimators':[100,200,300], 'max_depth':[2,3,4],
     'learning_rate':[0.03,0.05,0.1], 'subsample':[0.8,1.0]},
    cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=0)
xgb_safe_gs.fit(X_tr_s, y_tr_s)

xgb_safe_best  = xgb_safe_gs.best_estimator_
y_pred_safe    = xgb_safe_best.predict(X_te_s)
safe_rmse      = np.sqrt(mean_squared_error(y_te_s, y_pred_safe)) * 100
safe_r2        = r2_score(y_te_s, y_pred_safe)
safe_mae       = mean_absolute_error(y_te_s, y_pred_safe) * 100

print("Safe-Feature Early-Intervention Model (XGBoost Tuned)")
print(f"  Features : {len(FEATURES_SAFE)} advance-available features (no ABS rates)")
print(f"  Test RMSE: {safe_rmse:.3f}%")
print(f"  Test MAE : {safe_mae:.3f}%")
print(f"  Test R²  : {safe_r2:.4f}")
print(f"\nFull model (9 features) Test RMSE : {xgb_te_rmse:.3f}%  R²: {xgb_te_r2:.4f}")
print(f"Safe model (7 features) Test RMSE : {safe_rmse:.3f}%  R²: {safe_r2:.4f}")
print(f"RMSE cost of removing ABS rates   : +{safe_rmse - xgb_te_rmse:.3f} pp")

In [ ]:
# ── Compare full vs safe model predictions ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, name, yp, yte_, color in [
    (axes[0], 'Full Model (9 features)', y_pred_xgb,  y_te,   TEAL),
    (axes[1], 'Safe Model (7 pre-year features)', y_pred_safe, y_te_s, AMBER),
]:
    ax.scatter(yte_*100, yp*100, color=color, alpha=0.5, s=35, edgecolors='none')
    mn = min(yte_.min(), yp.min())*100 - 1
    mx = max(yte_.max(), yp.max())*100 + 1
    ax.plot([mn,mx],[mn,mx], color='white', lw=1.5, ls='--', alpha=0.5, label='Perfect fit')
    r2_  = r2_score(yte_, yp)
    rmse_= np.sqrt(mean_squared_error(yte_, yp))*100
    ax.set_xlabel('Actual (%)'); ax.set_ylabel('Predicted (%)')
    ax.set_title(f'{name}\nR²={r2_:.4f}   RMSE={rmse_:.3f} pp')
    ax.legend(fontsize=9); ax.set_xlim(mn,mx); ax.set_ylim(mn,mx)

plt.suptitle('Full Model vs Safe-Feature Early-Intervention Model', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

# ── Safe model feature importance ─────────────────────────────────────────
safe_labels = ['FRL%','Log(Enroll)','HS%','Urban','SizeTier','FRL\u00b2','Year']
safe_imp    = xgb_safe_best.feature_importances_
order_s     = np.argsort(safe_imp)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([safe_labels[i] for i in order_s], [safe_imp[i] for i in order_s],
        color=AMBER, alpha=0.85, height=0.55, edgecolor='none')
for i, idx in enumerate(order_s):
    ax.text(safe_imp[idx]+0.005, i, f'{safe_imp[idx]*100:.1f}%',
            va='center', fontsize=9, color='white')
ax.set_xlabel('Feature Importance')
ax.set_title('Safe-Feature Model — Importance\n(FRL% is #1 advance-available predictor)')
plt.tight_layout(); plt.show()

**Safe-Feature Model Findings:**

Without ABS10/ABS15 (mid-year measurements), the safe model relies entirely on pre-year structural features. `FRL_PCT` becomes the dominant predictor — confirming it is the best advance signal a district administrator has before the school year begins.

The accuracy cost of removing ABS rates is the price of making the model deployable as a **true early-warning system**. The safe model can be run at enrollment finalization each August, generating a district risk score for NDE to use in resource allocation.

## 13. Per-Year Generalization Check

XGBoost (Tuned) evaluated separately on each of the 7 test-year subsets. Confirms the model generalizes across pre-COVID, COVID-peak, and recovery years.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes_flat = axes.flatten()

for i, yr in enumerate(ALL_YEARS):
    ax   = axes_flat[i]
    mask = yr_te == yr
    if mask.sum() == 0:
        ax.axis('off'); continue
    yt_yr = y_te[mask]; yp_yr = y_pred_xgb[mask]
    r2_yr   = r2_score(yt_yr, yp_yr)
    rmse_yr = np.sqrt(mean_squared_error(yt_yr, yp_yr)) * 100
    ax.scatter(yt_yr*100, yp_yr*100, color=YEAR_COLORS[yr], alpha=0.75, s=50, edgecolors='none')
    mn = min(yt_yr.min(), yp_yr.min())*100 - 1
    mx = max(yt_yr.max(), yp_yr.max())*100 + 1
    ax.plot([mn,mx],[mn,mx], color='white', lw=1.2, ls='--', alpha=0.5)
    ax.set_xlabel('Actual (%)'); ax.set_ylabel('Predicted (%)')
    ax.set_title(f'{yr}\nR\u00b2={r2_yr:.3f}  RMSE={rmse_yr:.2f}%  n={mask.sum()}')
    ax.set_xlim(mn,mx); ax.set_ylim(mn,mx)

axes_flat[-1].axis('off')
plt.suptitle('XGBoost Tuned \u2014 Per-Year Accuracy on Test Set', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 14. Final Summary & Report Numbers

All numbers below are the definitive figures for the written final report.

In [ ]:
print("=" * 72)
print("DISTRICT-LEVEL ML ANALYSIS — FINAL RESULTS (Abhinav Adhikari)")
print("=" * 72)
print(f"{'Model':<28} {'Train RMSE':>12} {'Test RMSE':>12} {'Train R²':>10} {'Test R²':>10}")
print("-" * 72)

all_models = [
    ('XGBoost — Default',    xgb_def.predict(X_te),   xgb_def.predict(X_tr)),
    ('XGBoost — Tuned ★',   y_pred_xgb,               xgb_best.predict(X_tr)),
    ('Random Forest — Default', rf_def.predict(X_te), rf_def.predict(X_tr)),
    ('Random Forest — Tuned',   y_pred_rf,             rf_best.predict(X_tr)),
    ('LightGBM — Default',   lgb_def.predict(X_te),   lgb_def.predict(X_tr)),
    ('LightGBM — Tuned',     y_pred_lgb,               lgb_best.predict(X_tr)),
]

for lbl, yp_te, yp_tr in all_models:
    rmse_te = np.sqrt(mean_squared_error(y_te, yp_te)) * 100
    rmse_tr = np.sqrt(mean_squared_error(y_tr, yp_tr)) * 100
    r2_te   = r2_score(y_te, yp_te)
    r2_tr   = r2_score(y_tr, yp_tr)
    print(f"  {lbl:<26} {rmse_tr:>11.3f}%  {rmse_te:>11.3f}%  {r2_tr:>9.4f}  {r2_te:>9.4f}")

print()
print("Safe-Feature Early-Intervention Model (XGBoost, 7 pre-year features):")
print(f"  Test RMSE: {safe_rmse:.3f}%  |  Test R²: {safe_r2:.4f}  |  Test MAE: {safe_mae:.3f}%")
print()
print(f"Dataset  : {len(ml)} district-year observations | {ml['DIST_KEY'].nunique()} unique districts | 7 years")
print(f"Split    : 80/20 stratified by year | seed=42")
print(f"Tuning   : GridSearchCV 5-fold CV on training set only | test set never touched")